# Lab 8 — Silver to Gold & SCD Type 2 Upserts

**Goal:** Build a Gold revenue mart and preserve customer history using SCD Type 2.

**Prerequisite:** Lab 7 Silver table. This notebook creates separate dimension/staging tables for repeatable practice.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_db.gold_daily_revenue_lab8
USING DELTA
AS
SELECT
  to_date(order_timestamp) AS order_date,
  customer_id,
  SUM(amount) AS total_revenue,
  COUNT(*) AS order_count
FROM workspace.lab_db.silver_orders_lab7
GROUP BY to_date(order_timestamp), customer_id;

SELECT * FROM workspace.lab_db.gold_daily_revenue_lab8
ORDER BY order_date, customer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_db.gold_dim_customers_lab8 (
  customer_id INT,
  name STRING,
  email STRING,
  effective_date DATE,
  end_date DATE,
  is_current BOOLEAN
)
USING DELTA;

INSERT INTO workspace.lab_db.gold_dim_customers_lab8 VALUES
  (101, 'Alice Smith', 'alice@example.com', DATE '2025-01-01', NULL, true),
  (102, 'Bob Jones', 'bob@example.com', DATE '2025-01-01', NULL, true);

SELECT * FROM workspace.lab_db.gold_dim_customers_lab8 ORDER BY customer_id;

## Stage changed customer records

We simulate an updated email for customer 101 and a new customer 103. The MERGE expires the current record for changed customers; a second MERGE inserts the new current versions.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_db.staging_customer_updates_lab8 (
  customer_id INT,
  name STRING,
  email STRING
)
USING DELTA;

INSERT INTO workspace.lab_db.staging_customer_updates_lab8 VALUES
  (101, 'Alice Smith', 'alice.new@example.com'),
  (103, 'Charlie Brown', 'charlie@example.com');

In [0]:
%sql
-- Expire current records where the incoming attributes differ
MERGE INTO workspace.lab_db.gold_dim_customers_lab8 AS target
USING workspace.lab_db.staging_customer_updates_lab8 AS source
ON target.customer_id = source.customer_id AND target.is_current = true
WHEN MATCHED AND (
  NOT (target.name <=> source.name) OR
  NOT (target.email <=> source.email)
)
THEN UPDATE SET
  target.is_current = false,
  target.end_date = current_date();

In [0]:
%sql
-- Insert current records for new customers or customers whose current row was just expired
INSERT INTO workspace.lab_db.gold_dim_customers_lab8
SELECT
  source.customer_id,
  source.name,
  source.email,
  current_date() AS effective_date,
  CAST(NULL AS DATE) AS end_date,
  true AS is_current
FROM workspace.lab_db.staging_customer_updates_lab8 source
LEFT JOIN workspace.lab_db.gold_dim_customers_lab8 target
  ON source.customer_id = target.customer_id AND target.is_current = true
WHERE target.customer_id IS NULL
   OR NOT (target.name <=> source.name)
   OR NOT (target.email <=> source.email);

In [0]:
%sql
SELECT * FROM workspace.lab_db.gold_dim_customers_lab8
ORDER BY customer_id, effective_date, is_current;

## Verification

Customer 101 should have an expired historical version and a new current version. Customer 102 should remain current. Customer 103 should have one current version.

In [0]:
%sql
SELECT customer_id, COUNT(*) AS versions,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_versions
FROM workspace.lab_db.gold_dim_customers_lab8
GROUP BY customer_id
ORDER BY customer_id;